[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter7/react-agent-langchain.ipynb)

# Chapter 7: ReAct Agent with LangChain

This notebook demonstrates building a ReAct (Reasoning + Acting) agent using LangChain that combines RAG retrieval with tool-augmented reasoning.

**What you'll learn:**
- Build a RAG chain over a PDF document using LangChain and FAISS
- Wrap the RAG chain as a tool for an agent using the `@tool` decorator
- Create a ReAct agent with LangGraph that reasons about when to use tools
- Stream agent reasoning steps to observe the Reasoning + Acting loop

**Prerequisites:** `pip install langchain langchain-openai langchain-community langgraph faiss-cpu pypdf` and set `OPENAI_API_KEY`.

## Setup

We import LangChain components for embeddings, vector search, prompt templates, and LangGraph's prebuilt ReAct agent.

In [1]:
import os
import warnings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool

# Suppress deprecation warning - the suggested import location doesn't exist yet
warnings.filterwarnings("ignore", message="create_react_agent has been moved")
from langgraph.prebuilt import create_react_agent

## Build RAG Chain

We load the GPT-2 paper, split it into chunks, embed them into FAISS, and wire up a retrieval-augmented generation chain using LCEL.

In [2]:
# 1. Initialize the LLM and Embeddings models
llm = ChatOpenAI(model="gpt-4o", temperature=0)
embeddings = OpenAIEmbeddings()

# 2. Load and Process the GPT-2 paper PDF document
loader = PyPDFLoader("language_models_are_unsupervised_multitask_learners.pdf")
docs = loader.load_and_split()

# Split the document into chunks for better processing
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)

# 3. Create a Vector Store (Knowledge Base) using FAISS
vectorstore = FAISS.from_documents(texts, embeddings)
retriever = vectorstore.as_retriever()

# 4. Create the RAG Chain using LCEL (LangChain Expression Language)
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

## Define Tools and Agent

We wrap the RAG chain as a callable tool using `@tool`, then hand it to a LangGraph ReAct agent that can reason about when retrieval is needed.

In [3]:
# 5. Define the RAG Tool for the Agent using @tool decorator
@tool
def rag_gpt_tool(question: str) -> str:
    """Use this tool to answer questions about the 'GPT-2' paper. 
    It can answer any question related to this topic from that original paper."""
    return rag_chain.invoke(question)

tools = [rag_gpt_tool]

# 6. Create the ReAct Agent using LangGraph
agent = create_react_agent(llm, tools)

## Run the Agent

We stream the agent's execution to see each Reasoning → Acting → Observation step as it decides which tool calls to make and synthesizes the final answer.

In [4]:
# 7. Run the Agent with verbose output to see ReAct steps
question = "What is the size of GPT-2 in terms of number of weights? How does that influence its performance?"

print("=" * 60)
print(f"Question: {question}")
print("=" * 60)

# Stream the agent's response to see reasoning steps
for chunk in agent.stream({"messages": [("user", question)]}):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][0]
        # Check if agent is calling a tool
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"\n🔧 Action: {tool_call['name']}")
                print(f"   Input: {tool_call['args']}")
        # Print agent's reasoning/response
        if msg.content:
            print(f"\n💭 Agent: {msg.content}")
    elif "tools" in chunk:
        tool_msg = chunk["tools"]["messages"][0]
        print(f"\n📋 Observation: {tool_msg.content}")

print("\n" + "=" * 60)
print("Done!")
print("=" * 60)

Question: What is the size of GPT-2 in terms of number of weights? How does that influence its performance?

🔧 Action: rag_gpt_tool
   Input: {'question': 'What is the size of GPT-2 in terms of number of weights?'}

🔧 Action: rag_gpt_tool
   Input: {'question': 'How does the size of GPT-2 influence its performance?'}

📋 Observation: The size of GPT-2 in terms of the number of weights (parameters) is 1,542 million (or 1.542 billion).

📋 Observation: The size of GPT-2, which has over an order of magnitude more parameters than the original GPT, significantly influences its performance. The increased model capacity allows GPT-2 to achieve state-of-the-art results on 7 out of 8 tested language modeling datasets in a zero-shot setting. It answers 5.3 times more questions correctly compared to the smallest model, indicating that model capacity has been a major factor in improving performance on tasks like reading comprehension. Additionally, GPT-2's probability assignments to its generated an